In [2]:
import geopandas as gpd

gdf = gpd.read_file("../data/raw/IGN/TRONCON_DE_ROUTE.shp")

print(f"Shape: {gdf.shape}")
print(f"\nColumns:\n{list(gdf.columns)}")
print(f"\nDtypes:\n{gdf.dtypes}")
print(f"\nFirst row:\n{gdf.iloc[0]}")
print(f"\nCRS: {gdf.crs}")

Shape: (493865, 87)

Columns:
['ID', 'NATURE', 'NOM_COLL_G', 'NOM_COLL_D', 'IMPORTANCE', 'FICTIF', 'POS_SOL', 'ETAT', 'DATE_CREAT', 'DATE_MAJ', 'DATE_APP', 'DATE_CONF', 'SOURCE', 'ID_SOURCE', 'ACQU_PLANI', 'PREC_PLANI', 'ACQU_ALTI', 'PREC_ALTI', 'NB_VOIES', 'LARGEUR', 'IT_VERT', 'PRIVE', 'SENS', 'BUS', 'URBAIN', 'VIT_MOY_VL', 'ACCES_VL', 'ACCES_PED', 'FERMETURE', 'NAT_RESTR', 'RESTR_H', 'RESTR_P', 'RESTR_PPE', 'RESTR_LAR', 'RESTR_LON', 'RESTR_MAT', 'BORNEDEB_G', 'BORNEDEB_D', 'BORNEFIN_G', 'BORNEFIN_D', 'INSEECOM_G', 'INSEECOM_D', 'ALIAS_G', 'ALIAS_D', 'DATE_SERV', 'ID_RN', 'ID_ITI', 'NUMERO', 'NUM_EUROP', 'CL_ADMIN', 'GESTION', 'TOPONYME', 'ITI_CYCL', 'VOIE_VERTE', 'NATURE_ITI', 'NOM_ITI', 'DELESTAGE', 'SRC_BAN_G', 'SRC_BAN_D', 'NOM_BAN_G', 'NOM_BAN_D', 'LD_BAN_G', 'LD_BAN_D', 'ID_BAN_G', 'ID_BAN_D', 'SENS_CYC_G', 'SENS_CYC_D', 'CYCLABLE_G', 'CYCLABLE_D', 'RETOURDFCI', 'GAB_DFCI', 'IMPAS_DFCI', 'NDET_DFCI', 'OALIM_DFCI', 'PTMAX_DFCI', 'PISTE_DFCI', 'DFCI_DEBRO', 'DFCI_FOSSE', 'SENS_DF

In [3]:
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

def load_ign_road_network(shp_path, precision=1):
    """
    Load IGN TRONCON_DE_ROUTE shapefile as a NetworkX graph.
    Nodes are built from edge endpoints with coordinate rounding to fix floating point mismatches.
    
    Args:
        shp_path: Path to TRONCON_DE_ROUTE.shp
        precision: Decimal places for coordinate rounding (default: 1)
    """
    gdf = gpd.read_file(shp_path)
    
    G = nx.Graph()
    node_coords = {}  # (x, y) -> node_id
    node_counter = 0

    def get_or_create_node(x, y):
        nonlocal node_counter
        key = (round(x, precision), round(y, precision))
        if key not in node_coords:
            node_coords[key] = node_counter
            G.add_node(node_counter, x=key[0], y=key[1])
            node_counter += 1
        return node_coords[key]

    for _, row in gdf.iterrows():
        coords = list(row.geometry.coords)
        x_start, y_start = coords[0][0], coords[0][1]
        x_end, y_end = coords[-1][0], coords[-1][1]

        u = get_or_create_node(x_start, y_start)
        v = get_or_create_node(x_end, y_end)

        if u != v:
            G.add_edge(u, v, 
                       id=row['ID'],
                       nature=row['NATURE'],
                       importance=row['IMPORTANCE'],
                       length=row.geometry.length)

    print(f"Nodes: {G.number_of_nodes():,}")
    print(f"Edges: {G.number_of_edges():,}")
    print(f"Components: {nx.number_connected_components(G):,}")
    largest = max(nx.connected_components(G), key=len)
    print(f"Largest component: {len(largest):,} nodes")

    return G

G_ign = load_ign_road_network("../data/raw/IGN/TRONCON_DE_ROUTE.shp")

Nodes: 388,148
Edges: 487,518
Components: 323
Largest component: 386,947 nodes


In [5]:
import pickle
import networkx as nx

with open("../data/processed/Dataset1/Graph.pkl", "rb") as f:
    G = pickle.load(f)

# Structure
print(f"Type: {type(G)}")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

# Node attributes
sample_node = list(G.nodes(data=True))[0]
print(f"\nSample node: {sample_node}")

# Edge attributes
sample_edge = list(G.edges(data=True))[0]
print(f"\nSample edge: {sample_edge}")

Type: <class 'networkx.classes.graph.Graph'>
Nodes: 43,826
Edges: 44,210

Sample node: (2787, {'id': 0, 'CODCOMM': '340256', 'COMMUNE': 'SAINT GENIES DES MOURGUES', 'TYPE': 'REGARD NON TYPE', 'TYPEREG': 'REGARD BLOQUE', 'NOM_VOIE': nan, 'TYPE_VOIE': nan, 'DATE_POSE': Timestamp('2999-12-31 00:00:00'), 'TAMP_ANOM': nan, 'NOTES': nan, 'DATE_REC': None, 'DEBITMETRE': None, 'MATERIAU': nan, 'DIAMETRE': nan, 'ID_TR': None, 'SOURCE': None, 'PROF_RADIE': '0', 'CADRE_ANOM': nan, 'ENTREPRISE': nan, 'FOURNISSEU': nan, 'MAJ': 'RAS', 'OBSERVATIO': None, 'NUM_VOIE': None, 'IMPLANT': nan, 'CAPACITE': nan, 'CUN_ANOM': nan, 'NBPOMPES': nan, 'B_NMTH': nan, 'B_NMH': nan, 'B_HMNORMAL': nan, 'B_NMB': nan, 'B_NMTB': nan, 'P1_MARQUE': nan, 'P1_TYPE': nan, 'P1_DEBIT': nan, 'P1_PUISSAN': nan, 'P1_HMT': nan, 'P2_MARQUE': nan, 'P2_TYPE': nan, 'P2_DEBIT': nan, 'P2_PUISSAN': nan, 'P2_HMT': nan, 'P3_MARQUE': nan, 'P3_TYPE': nan, 'P3_PUISSAN': nan, 'P3_DEBIT': nan, 'P3_HMT': nan, 'PARTICULA': nan, 'PRO_RADIER': '0',

In [1]:
import json

def simplify_geojson(nodes_path, pipes_path, out_nodes_path, out_pipes_path):
    # Nodes: keep id, old_id, source_1 + geometry
    with open(nodes_path) as f:
        nodes = json.load(f)
    for feature in nodes["features"]:
        p = feature["properties"]
        feature["properties"] = {
            "id": p["id"],
            "old_id": p["old_id"],
            "source_1": p["source_1"]
        }
    with open(out_nodes_path, "w") as f:
        json.dump(nodes, f)

    # Pipes: keep id, old_id, sourceNode, targetNode + geometry
    with open(pipes_path) as f:
        pipes = json.load(f)
    for feature in pipes["features"]:
        p = feature["properties"]
        feature["properties"] = {
            "id": p["id"],
            "old_id": p["old_id"],
            "sourceNode": p["sourceNode"],
            "targetNode": p["targetNode"]
        }
    with open(out_pipes_path, "w") as f:
        json.dump(pipes, f)

In [14]:
nodes_path = "../data/raw/Prades/Dataset5/Nodes.geojson"
pipes_path = "../data/raw/Prades/Dataset5/Pipes.geojson"
out_nodes_path =  "../data/raw/Prades/Dataset5/Nodes.geojson"
out_pipes_path = "../data/raw/Prades/Dataset5/Pipes.geojson"

In [15]:
simplify_geojson(nodes_path, pipes_path, out_nodes_path, out_pipes_path)